
# 12 — Data Imputation and Encoding

**Scope:** The missing and incorrect values are handled, categorical features are encoded.


Since we decided to predict missing and incorrect values, we have to split the training data into training and validation sets before applying imputation and encoding.

# Table of Contents

Take this as an example for a Table of Contents for your notebook.
We have to fix all the names and sections according to what we actually do in the notebook.

<a class="anchor" id="top"></a>

** **

1. [Importing Libraries & Data](#1.-Importing-Libraries-&-Data) <br><br>
    
2. [Exploratory Data Analysis](#2.-Exploratory-Data-Analysis)
    
   2.1 [Incoherencies](#2.1-Incoherencies) <br>
   
   &emsp; 2.1.1 [Address Identified Incoherencies](#2.1.1-Address-Identified-Incoherencies) <br><br>
    
3. [Data Cleaning & Preprocessing](#3.-Data-Cleaning-&-Preprocessing)

   3.1 [Duplicates](#3.1-Duplicates) <br>
    
   3.2 [Feature Engineering](#3.2-Feature-Engineering) <br>
   
   &emsp; 3.2.1 [Data Type Conversions](#3.2.1-Data-Type-Conversions) <br>
   
   &emsp; 3.2.2 [Encoding](#3.2.2-Encoding) <br>
   
   &emsp; 3.2.3 [Other Transformations](#3.2.3-Other-Transformations) <br>
    
   &emsp; 3.2.4 [Unique Feature-Pair Analysis](#3.2.4-Unique-Feature-Pair-Analysis) <br> 

   3.3 [Train-Test Split](#3.3-Train-Test-Split) <br>
   
   3.4 [Missing Values](#3.4-Missing-Values) <br>
    
   3.5 [Outliers](#3.5-Outliers) <br>

   3.6 [Visualisations](#3.6-Visualisations) <br><br>
   

In [344]:
import os, re, math, warnings
from pathlib import Path
from datetime import datetime
import json
import pandas as pd
import numpy as np
import re
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import RobustScaler

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
pd.set_option("mode.copy_on_write", True)
warnings.filterwarnings("ignore")

RANDOM_STATE = 42  # for reproducibility of any sampling


In [289]:
# Load the data paths
data_dir = "../data/"

# Load the raw data into a pandas dataframe
df = pd.read_csv(os.path.join(data_dir, "processed_data/11_processed_train_data.csv"))
X_test = pd.read_csv(os.path.join(data_dir, "test.csv"))

# Drop the column carID from both dataframes because it is not needed for modeling
if "carID" in df.columns:
    df = df.drop(columns=["carID"])
if "carID" in X_test.columns:
    x_test = X_test.drop(columns=["carID"])


print("Loaded shape:", df.shape)
display(df.head(3))

print("Loaded test shape:", X_test.shape)
display(X_test.head(3))


Loaded shape: (75973, 13)


,Brand,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
0,Volkswagen,Golf,2016.0,22290.0,Semi-Auto,28421.0,Petrol,NaN,11.417268,2.0,63.0,4.0,0.0
1,Toyota,Yaris,2019.0,13790.0,Manual,4589.0,Petrol,145.0,47.900000,1.5,50.0,1.0,0.0
2,Audi,Q2,2019.0,24990.0,Semi-Auto,3624.0,Petrol,145.0,40.900000,1.5,56.0,4.0,0.0


Loaded test shape: (32567, 13)


,carID,Brand,model,year,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
0,89856,Hyundai,I30,2022.878006,Automatic,30700.000000,petrol,205.0,41.5,1.6,61.0,3.0,0.0
1,106581,VW,Tiguan,2017.000000,Semi-Auto,-48190.655673,Petrol,150.0,38.2,2.0,60.0,2.0,0.0
2,80886,BMW,2 Series,2016.000000,Automatic,36792.000000,Petrol,125.0,51.4,1.5,94.0,2.0,0.0


### Split the Training data into training and validation sets

In [290]:
# ======================================================
# Split df into Training and Validation Set (60/20/20 total)
# ======================================================

from sklearn.model_selection import train_test_split

# Separate features and target from the training data
X = df.drop(columns=["price"])
y = df["price"]

# Split df (80% of total data) into training and validation
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.25,        # 25% of 80% train = 20% of total
    random_state=42,
)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_val shape:   {X_val.shape}")
print(f"y_val shape:   {y_val.shape}")
print(f"X_test shape:  {X_test.shape}")


X_train shape: (56979, 12)
y_train shape: (56979,)
X_val shape:   (18994, 12)
y_val shape:   (18994,)
X_test shape:  (32567, 13)


# Handling Missing Values

- Numerical: fill with the median
- Categorical: fill with the most frequent value

In [291]:
import pandas as pd

def missing_report(X: pd.DataFrame, name: str) -> pd.DataFrame:
    mv = X.isna().sum()
    mv = mv[mv > 0].sort_values(ascending=False)
    if mv.empty:
        print(f"[{name}] No missing values found. (n_rows={len(X)})")
        return pd.DataFrame(columns=["n_missing", "pct_missing"])
    pct = (mv / len(X) * 100).round(2)
    report = pd.DataFrame({"n_missing": mv, "pct_missing": pct})
    print(f"[{name}] Missing Values (n_rows={len(X)}):")
    display(report)
    return report

# Reports for the splits
mv_train = missing_report(X_train, "X_train")
mv_val   = missing_report(X_val,   "X_val")
mv_test  = missing_report(X_test,  "X_test")

# Helpful for the next step (imputation/encoding):
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print(f"#numeric_cols: {len(numeric_cols)} → {numeric_cols[:10]}{' ...' if len(numeric_cols) > 10 else ''}")
print(f"#categorical_cols: {len(categorical_cols)} → {categorical_cols[:10]}{' ...' if len(categorical_cols) > 10 else ''}")


[X_train] Missing Values (n_rows=56979):


,n_missing,pct_missing
mpg,6945,12.19
tax,6251,10.97
engineSize,1583,2.78
previousOwners,1412,2.48
mileage,1372,2.41
model,1253,2.20
transmission,1147,2.01
paintQuality%,1141,2.00
hasDamage,1141,2.00
year,1127,1.98


[X_val] Missing Values (n_rows=18994):


,n_missing,pct_missing
mpg,2246,11.82
tax,2031,10.69
previousOwners,509,2.68
engineSize,508,2.67
mileage,460,2.42
model,435,2.29
fuelType,416,2.19
hasDamage,407,2.14
paintQuality%,383,2.02
transmission,375,1.97


[X_test] Missing Values (n_rows=32567):


,n_missing,pct_missing
tax,3308,10.16
mpg,3288,10.10
mileage,689,2.12
fuelType,656,2.01
year,653,2.01
model,650,2.00
Brand,649,1.99
engineSize,628,1.93
paintQuality%,625,1.92
transmission,623,1.91


#numeric_cols: 8 → ['year', 'mileage', 'tax', 'mpg', 'engineSize', 'paintQuality%', 'previousOwners', 'hasDamage']
#categorical_cols: 4 → ['Brand', 'model', 'transmission', 'fuelType']


Why we chose random forest for imputation of 'transmission' feature? 

tbd

In [292]:
def impute_transmission(df, min_model_count=10):
    # Make a copy so the original dataframe isn't modified directly
    df = df.copy()
    # Count how many entries exist per model
    # We drop rows where with nan for this count, because we dont want data leakage
    df_nona = df.copy().dropna()
    model_counts = df_nona['model'].value_counts()

    # Keep only models with enough data points
    valid_models = model_counts[model_counts >= min_model_count].index

    # Loop through those models and fill NaN with the most common value (mode)
    for model in valid_models:
        mask = (df['model'] == model) & (df['transmission'].isna())
        mode_values = df.loc[df['model'] == model, 'transmission'].mode()
        if not mode_values.empty:
            df.loc[mask, 'transmission'] = mode_values[0]

    # Check how many missing values are still left
    remaining_nas = df['transmission'].isna().sum()
    
    if remaining_nas > 0:
        # Encode categorical variables numerically
        # RandomForest can only work with numeric data
        for col in ['Brand', 'model', 'fuelType']:
            means = df.loc[df['transmission'].notna()].groupby(col)['price'].mean()
            df[col] = df[col].map(means) # for test data purposes, we only use infos from the traindata

        # Split into training (known transmission) and test (missing transmission)
        transmission_train = df[df['transmission'].notna()]
        transmission_test = df[df['transmission'].isna()]

        # Select predictor features
        features = ['Brand', 'model', 'fuelType', 'engineSize', 'year', 'mpg', 'tax']
        X_train = transmission_train[features]
        y_train = transmission_train['transmission']
        X_test = transmission_test[features]

        # Train a Random Forest classifier to predict transmission type
        model = RandomForestClassifier(
            n_estimators=200,  # number of trees
            max_depth=10,      # limit depth to avoid overfitting
            random_state=42
        )
        model.fit(X_train, y_train)

        # Predict missing transmission values
        preds = model.predict(X_test)
        df.loc[df['transmission'].isna(), 'transmission'] = preds
        return df

In [293]:
def impute_fuelType(df, min_model_count=10, is_test_data=True):
    
    # work on a copy
    df = df.copy()

    # per-model mode fill (only for models with enough rows)
    # We drop rows where with nan for this count, because we dont want data leakage
    df_nona = df.copy().dropna()
    model_counts = df_nona['model'].value_counts()
    valid_models = model_counts[model_counts >= min_model_count].index

    for m in valid_models:
        mask_missing = (df['model'] == m) & (df['fuelType'].isna())
        mode_vals = df.loc[df['model'] == m, 'fuelType'].mode()
        if not mode_vals.empty:
            df.loc[mask_missing, 'fuelType'] = mode_vals[0]

    # if still missing, train a classifier
    remaining = df['fuelType'].isna().sum()
    if remaining > 0:
        # encode categorical predictors
        if is_test_data:
            predictors = ['Brand', 'model', 'transmission', 'engineSize', 'year', 'mpg', 'price']
        else:
            predictors = ['Brand', 'model', 'transmission', 'engineSize', 'year', 'mpg']
        for col in predictors:
            if df[col].dtype == 'object':
                means = df.loc[df['fuelType'].notna()].groupby(col)['price'].mean()
                df[col] = df[col].map(means) # for test data purposes, we only use infos from the traindata

        # split into known vs missing target
        fuelType_train = df[df['fuelType'].notna()]
        test  = df[df['fuelType'].isna()]

        X_train = fuelType_train[predictors]
        y_train = fuelType_train['fuelType']
        X_test  = test[predictors]

        # encode target
        y_le = LabelEncoder()
        y_train_enc = y_le.fit_transform(y_train.astype(str))

        # train classifier
        clf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=77)
        clf.fit(X_train, y_train_enc)

        # predict and inverse-transform
        preds_enc = clf.predict(X_test)
        preds = y_le.inverse_transform(preds_enc)

        # fill back
        df.loc[df['fuelType'].isna(), 'fuelType'] = preds

    return df

In [294]:
def impute_numeric_rf(df, target_col, rf_features):
    """
    Fill missing numeric values in a column using a RandomForestRegressor.
    """

    # Copy the DataFrame so the original one is not changed
    df = df.copy()

    # If there are no missing values, just return the DataFrame
    if not df[target_col].isna().any():
        return df

    # Encode categorical features so the model can handle them
    for col in rf_features:
        if df[col].dtype == 'object':
            means = df.loc[df[target_col].notna()].groupby(col)['price'].mean()
            df[col] = df[col].map(means) # for test data purposes, we only use infos from the traindata

    # Split data into known and missing target values
    train = df[df[target_col].notna()]
    test = df[df[target_col].isna()]

    X_train = train[rf_features]
    y_train = train[target_col]
    X_test = test[rf_features]

    # Train the Random Forest model
    # n_estimators=300 number of trees in the forest, more trees → better accuracy, but slower
    # max_depth=12 maximum depth of each tree, limits how detailed (complex) each tree can get → prevents overfitting
    # random_state=23 seed for randomness, makes results reproducible (same every time you run it)
    model = RandomForestRegressor(n_estimators=300, max_depth=12, random_state=23)
    model.fit(X_train, y_train)

    # Predict missing values and fill them in
    preds = model.predict(X_test)
    df.loc[df[target_col].isna(), target_col] = preds

    return df

In [295]:
def impute_mileage(df):
    # Make a copy so the original DataFrame is not modified
    df = df.copy()

    # Calculate the mean mileage for each year (only using non-missing values)
    year_medians = df.groupby('year')['mileage'].median()

    # Replace missing mileage values with the corresponding year mean
    df['mileage'] = df['mileage'].fillna(df['year'].map(year_medians))
    df['mileage'] = df['mileage'].fillna(df['mileage'].median())
    # Return the DataFrame with imputed values
    return df

In [296]:
def impute_year(df, bins=30):

    # Make a copy to avoid modifying the original DataFrame
    df = df.copy()

    # Create mileage bins
    df['mileage_bin'] = pd.cut(df['mileage'], bins=bins)

    # Calculate mean year per mileage bin (only where year is known)
    bin_medians = df.groupby('mileage_bin')['year'].median().round()

    # Map each row’s bin to its mean year
    df['year'] = df['year'].fillna(df['mileage_bin'].map(bin_medians))

    df['year'] = df['year'].fillna(df['year'].median())

    # Drop the helper column
    df.drop(columns='mileage_bin', inplace=True)

    return df

In [297]:
def impute_mpg(df):
    # Make a copy so the original DataFrame is not modified
    df = df.copy()

     # Calculate median mpg by (model, fuelType)
    model_medians = df.groupby(['model', 'fuelType'])['mpg'].median()

    # Calculate fallback median mpg by (Brand, fuelType)
    brand_medians = df.groupby(['Brand', 'fuelType'])['mpg'].median()

    # Calculate fallback median mpg by fuelType
    fuelType_medians = df.groupby(['fuelType'])['mpg'].median()

    # Iterate through missing rows and fill based on available group
    for idx, row in df[df['mpg'].isna()].iterrows():
        key_model = (row['model'], row['fuelType'])
        key_brand = (row['Brand'], row['fuelType'])

        if key_model in model_medians.index:
            df.at[idx, 'mpg'] = model_medians.loc[key_model]
        elif key_brand in brand_medians.index:
            df.at[idx, 'mpg'] = brand_medians.loc[key_brand]
    df['mpg'] = df['mpg'].fillna(df['fuelType'].map(fuelType_medians))
    df['mpg'] = df['mpg'].fillna(df['mpg'].median())

    # Return the DataFrame with imputed values
    return df

In [298]:
def impute_paintQuality(df):

    df = df.copy()

    median_val = df['paintQuality%'].median()
    df['paintQuality%'] = df['paintQuality%'].fillna(median_val)    

    return df

In [299]:
def impute_previousOwners(df):

    df = df.copy()

    median_val = df['previousOwners'].median()
    df['previousOwners'] = df['previousOwners'].fillna(median_val)    

    return df

In [300]:
def impute_tax(df, bins=15):
    # Make a copy so the original DataFrame is not modified
    df = df.copy()
    df['mpg_bin'] = pd.cut(df['mpg'], bins=bins)
    
     # Calculate median tax by (model, fuelType)
    medians_one = df.groupby(['model', 'fuelType', 'mpg_bin'])['tax'].median()

    # Calculate fallback median tax by (Brand, fuelType)
    medians_two = df.groupby(['Brand', 'fuelType', 'mpg_bin'])['tax'].median()

    # Calculate fallback median tax by fuelType
    medians_three = df.groupby(['fuelType', 'mpg_bin'])['tax'].median()

    # Iterate through missing rows and fill based on available group
    for idx, row in df[df['tax'].isna()].iterrows():
        key_model = (row['model'], row['fuelType'], row['mpg_bin'])
        key_brand = (row['Brand'], row['fuelType'], row['mpg_bin'])
        key_else =  (row['fuelType'], row['mpg_bin'])

        if key_model in medians_one.index:
            df.at[idx, 'tax'] = medians_one.loc[key_model]
        elif key_brand in medians_two.index:
            df.at[idx, 'tax'] = medians_two.loc[key_brand]
        else:
            df.at[idx, 'tax'] = medians_three.loc[key_else]
    fuelType_medians = df.groupby('fuelType')['tax'].median()

    # Replace missing mileage values with the corresponding year mean
    df['tax'] = df['tax'].fillna(df['fuelType'].map(fuelType_medians))
    df['tax'] = df['tax'].fillna(df['tax'].median())

    # Return the DataFrame with imputed values
    return df

In [301]:
def impute_engineSize(df):
    # Make a copy so the original DataFrame is not modified
    df = df.copy()

     # Calculate median mpg by (model, fuelType)
    model_medians = df.groupby(['model', 'fuelType'])['engineSize'].median()

    # Calculate fallback median mpg by (Brand, fuelType)
    brand_medians = df.groupby(['Brand', 'fuelType'])['engineSize'].median()

    # Calculate fallback median mpg by fuelType
    fuelType_medians = df.groupby(['model'])['engineSize'].median()

    # Iterate through missing rows and fill based on available group
    for idx, row in df[df['engineSize'].isna()].iterrows():
        key_model = (row['model'], row['fuelType'])
        key_brand = (row['Brand'], row['fuelType'])

        if key_model in model_medians.index:
            df.at[idx, 'engineSize'] = model_medians.loc[key_model]
        elif key_brand in brand_medians.index:
            df.at[idx, 'engineSize'] = brand_medians.loc[key_brand]
    df['engineSize'] = df['engineSize'].fillna(df['fuelType'].map(fuelType_medians))
    df['engineSize'] = df['engineSize'].fillna(df['engineSize'].median())
    # Return the DataFrame with imputed values
    return df

In [302]:
def impute_brand (df, min_model_count=20, inlucePrice=True):
    
    # work on a copy
    df = df.copy()

    # Mapping: Modell → häufigste Marke
    df2=df.copy()
    model_to_brand = (
        df2.dropna(subset=['Brand', 'model'])
          .groupby('model')['Brand']
          .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
    )

    # Fehlende Marken füllen, wo Modell bekannt ist
    # sichere Kopie der Masken-Logik, NUR mit df2
    mask = df['Brand'].isna() & df['model'].notna()

    # benutze df2 für das Mapping (niemals df direkt!)
    df.loc[mask, 'Brand'] = df2.loc[mask, 'model'].map(model_to_brand)

    # if still missing, train a classifier
    remaining = df['Brand'].isna().sum()
    if remaining > 0:
        # encode categorical predictors
        if inlucePrice:
            predictors = ['transmission', 'engineSize', 'fuelType', 'mpg', 'price']
        else:
            predictors = ['transmission', 'engineSize', 'fuelType', 'mpg']
        for col in predictors:
            if df[col].dtype == 'object':
                means = df.loc[df["Brand"].notna()].groupby(col)['price'].mean()
                df[col] = df[col].map(means) # for test data purposes, we only use infos from the traindata

        # split into known vs missing target
        fuelType_train = df[df['Brand'].notna()]
        test  = df[df['Brand'].isna()]

        X_train = fuelType_train[predictors]
        y_train = fuelType_train['Brand']
        X_test  = test[predictors]

        # encode target
        y_le = LabelEncoder()
        y_train_enc = y_le.fit_transform(y_train.astype(str))

        # train classifier
        clf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=77)
        clf.fit(X_train, y_train_enc)

        # predict and inverse-transform
        preds_enc = clf.predict(X_test)
        preds = y_le.inverse_transform(preds_enc)

        # fill back
        df.loc[df['Brand'].isna(), 'Brand'] = preds

    return df

In [303]:
def impute_model(df, min_brand_count=20, includePrice=True):
    # Work on a copy so the original DataFrame is not modified
    df = df.copy()

    
    # --- Train a RandomForest classifier if there are still missing values ---
    # if still missing, train a classifier
    mode_lookup = (
    df.dropna(subset=['Brand', 'transmission', 'model'])
      .groupby(['Brand', 'transmission'])['model']
      .agg(lambda s: s.value_counts().idxmax())
    )
     
    for idx, row in df[df['model'].isna()].iterrows():
        brand = row['Brand']
        trans = row['transmission']
        key = (brand, trans)

        if key in mode_lookup.index:
            # häufigstes Modell einsetzen
            new_value = mode_lookup.loc[key]
            df.at[idx, 'model'] = new_value

    
    remaining = df['model'].isna().sum()
    
    if remaining > 0:
        # encode categorical predictors
        if includePrice:
            predictors = ['Brand', 'year', 'engineSize', 'mpg', 'tax', 'mileage', 'fuelType', 'transmission', 'price']
        else:
            predictors = ['Brand', 'year', 'engineSize', 'mpg', 'tax', 'mileage', 'fuelType', 'transmission']
        for col in predictors:
            if df[col].dtype == 'object':
                means = df.loc[df["model"].notna()].groupby(col)['price'].mean()
                df[col] = df[col].map(means) # for test data purposes, we only use infos from the traindata

        # split into known vs missing target
        fuelType_train = df[df['model'].notna()]
        test  = df[df['model'].isna()]

        X_train = fuelType_train[predictors]
        y_train = fuelType_train['model']
        X_test  = test[predictors]

        # encode target
        y_le = LabelEncoder()
        y_train_enc = y_le.fit_transform(y_train.astype(str))

        # train classifier
        clf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=77)
        clf.fit(X_train, y_train_enc)

        # predict and inverse-transform
        preds_enc = clf.predict(X_test)
        preds = y_le.inverse_transform(preds_enc)

        # fill back
        df.loc[df['model'].isna(), 'model'] = preds

    return df

In [304]:
%%time
X_train['price'] = y_train
X_train["transmission"] = impute_transmission(X_train)["transmission"]
X_train["fuelType"] = impute_fuelType(X_train)["fuelType"]
X_train["engineSize"] = impute_engineSize(X_train)["engineSize"]
X_train["mpg"] = impute_mpg(X_train)["mpg"]
X_train["tax"] = impute_tax(X_train)["tax"]
X_train["year"] = impute_year(X_train)["year"].round().astype(int)
X_train["mileage"] = impute_mileage(X_train)["mileage"]
X_train["paintQuality%"] = impute_paintQuality(X_train)["paintQuality%"]
X_train["previousOwners"] = impute_previousOwners(X_train)["previousOwners"]
X_train["Brand"] = impute_brand(X_train)["Brand"]
X_train["hasDamage"] = X_train["hasDamage"].fillna(1)
X_train["model"] = impute_model(X_train)["model"]
X_train = X_train.drop(columns=["price"])

CPU times: total: 27.2 s
Wall time: 27.5 s


In [305]:
# quick check
# Look at the Missing values
missing_values = X_train.isnull().sum()

print("Missing values in train:")
print(missing_values[missing_values > 0]) 
print("\n")

Missing values in train:
Series([], dtype: int64)




In [306]:
train_info = X_train.copy()
train_info["price"] = y_train
train_info["is_train"] = True
train_info = train_info.dropna()
train_info["row_id"] = np.nan
X_val["is_train"]=False
X_val["price"]=np.nan
X_val["row_id"]=X_val.index
X_val.info()
train_info.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18994 entries, 69512 to 66836
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Brand           18968 non-null  object 
 1   model           18559 non-null  object 
 2   year            18630 non-null  float64
 3   transmission    18619 non-null  object 
 4   mileage         18534 non-null  float64
 5   fuelType        18578 non-null  object 
 6   tax             16963 non-null  float64
 7   mpg             16748 non-null  float64
 8   engineSize      18486 non-null  float64
 9   paintQuality%   18611 non-null  float64
 10  previousOwners  18485 non-null  float64
 11  hasDamage       18587 non-null  float64
 12  is_train        18994 non-null  bool   
 13  price           0 non-null      float64
 14  row_id          18994 non-null  int64  
dtypes: bool(1), float64(9), int64(1), object(4)
memory usage: 2.2+ MB
<class 'pandas.core.frame.DataFrame'>
Index: 56979 entri

In [307]:
%%time
# compute transmission
compute_transmission = pd.concat([train_info.copy(), X_val.loc[X_val["transmission"].isna()]])
transmission_values = impute_transmission(compute_transmission).loc[compute_transmission["is_train"]== False]
for idx, row in X_val[X_val['transmission'].isna()].iterrows():
    row_id = row['row_id']
    new_value = transmission_values.loc[transmission_values['row_id']==row_id, 'transmission'].iloc[0]
    X_val.at[idx, 'transmission'] = new_value

CPU times: total: 4.78 s
Wall time: 4.8 s


In [308]:
%%time
# compute fuelType
compute_fuelType = pd.concat([train_info.copy(), X_val.loc[X_val["fuelType"].isna()]])
fuelType_values = impute_fuelType(compute_fuelType).loc[compute_fuelType["is_train"]== False]
for idx, row in X_val[X_val['fuelType'].isna()].iterrows():
    row_id = row['row_id']
    new_value = fuelType_values.loc[fuelType_values['row_id']==row_id, 'fuelType'].iloc[0]
    X_val.at[idx, 'fuelType'] = new_value

CPU times: total: 6.61 s
Wall time: 6.67 s


In [309]:
%%time
# compute engineSize
compute_engineSize = pd.concat([train_info.copy(), X_val.loc[X_val["engineSize"].isna()]])
engineSize_values = impute_engineSize(compute_engineSize).loc[compute_engineSize["is_train"]== False]
for idx, row in X_val[X_val['engineSize'].isna()].iterrows():
    row_id = row['row_id']
    new_value = engineSize_values.loc[engineSize_values['row_id']==row_id, 'engineSize'].iloc[0]
    X_val.at[idx, 'engineSize'] = new_value

CPU times: total: 328 ms
Wall time: 301 ms


In [310]:
# fill has damage
X_val["hasDamage"] = X_val["hasDamage"].fillna(1)

In [311]:
%%time
# compute mpg
compute_mpg = pd.concat([train_info.copy(), X_val.loc[X_val["mpg"].isna()]])
mpg_values = impute_mpg(compute_mpg).loc[compute_mpg["is_train"]== False]
for idx, row in X_val[X_val['mpg'].isna()].iterrows():
    row_id = row['row_id']
    new_value = mpg_values.loc[mpg_values['row_id']==row_id, 'mpg'].iloc[0]
    X_val.at[idx, 'mpg'] = new_value

CPU times: total: 1.06 s
Wall time: 1.12 s


In [312]:
%%time
# compute tax
compute_tax = pd.concat([train_info.copy(), X_val.loc[X_val["tax"].isna()]])
tax_values = impute_tax(compute_tax).loc[compute_tax["is_train"]== False]
for idx, row in X_val[X_val['tax'].isna()].iterrows():
    row_id = row['row_id']
    new_value = tax_values.loc[tax_values['row_id']==row_id, 'tax'].iloc[0]
    X_val.at[idx, 'tax'] = new_value

CPU times: total: 1.2 s
Wall time: 1.22 s


In [313]:
%%time
# compute year
compute_year = pd.concat([train_info.copy(), X_val.loc[X_val["year"].isna()]])
year_values = impute_year(compute_year).loc[compute_year["is_train"]== False]
for idx, row in X_val[X_val['year'].isna()].iterrows():
    row_id = row['row_id']
    new_value = year_values.loc[year_values['row_id']==row_id, 'year'].iloc[0]
    X_val.at[idx, 'year'] = new_value

CPU times: total: 156 ms
Wall time: 149 ms


In [314]:
%%time
# compute mileage
compute_mileage = pd.concat([train_info.copy(), X_val.loc[X_val["mileage"].isna()]])
mileage_values = impute_mileage(compute_mileage).loc[compute_mileage["is_train"]== False]
for idx, row in X_val[X_val['mileage'].isna()].iterrows():
    row_id = row['row_id']
    new_value = mileage_values.loc[mileage_values['row_id']==row_id, 'mileage'].iloc[0]
    X_val.at[idx, 'mileage'] = new_value

CPU times: total: 172 ms
Wall time: 171 ms


In [315]:
%%time
# compute paintQuality%
compute_paintQuality = pd.concat([train_info.copy(), X_val.loc[X_val["paintQuality%"].isna()]])
paintQuality_values = impute_paintQuality(compute_paintQuality).loc[compute_paintQuality["is_train"]== False]
for idx, row in X_val[X_val['paintQuality%'].isna()].iterrows():
    row_id = row['row_id']
    new_value = paintQuality_values.loc[paintQuality_values['row_id']==row_id, 'paintQuality%'].iloc[0]
    X_val.at[idx, 'paintQuality%'] = new_value

CPU times: total: 141 ms
Wall time: 150 ms


In [316]:
%%time
# compute previousOwners
compute_previousOwners = pd.concat([train_info.copy(), X_val.loc[X_val["previousOwners"].isna()]])
previousOwners_values = impute_previousOwners(compute_previousOwners).loc[compute_previousOwners["is_train"]== False]
for idx, row in X_val[X_val['previousOwners'].isna()].iterrows():
    row_id = row['row_id']
    new_value = previousOwners_values.loc[previousOwners_values['row_id']==row_id, 'previousOwners'].iloc[0]
    X_val.at[idx, 'previousOwners'] = new_value

CPU times: total: 203 ms
Wall time: 190 ms


In [317]:
%%time
# compute Brand
compute_brand = pd.concat([train_info.copy(), X_val.loc[X_val["Brand"].isna()]])
brand_values = impute_brand(compute_brand).loc[compute_brand["is_train"]== False]
for idx, row in X_val[X_val['Brand'].isna()].iterrows():
    row_id = row['row_id']
    new_value = brand_values.loc[brand_values['row_id']==row_id, 'Brand'].iloc[0]
    X_val.at[idx, 'Brand'] = new_value

CPU times: total: 6.44 s
Wall time: 6.46 s


In [318]:
%%time
# compute model
compute_model = pd.concat([train_info.copy(), X_val.loc[X_val["model"].isna()]])
model_values = impute_model(compute_model).loc[compute_model["is_train"]== False]
for idx, row in X_val[X_val['model'].isna()].iterrows():
    row_id = row['row_id']
    new_value = model_values.loc[model_values['row_id']==row_id, 'model'].iloc[0]
    X_val.at[idx, 'model'] = new_value

CPU times: total: 281 ms
Wall time: 260 ms


In [319]:
X_val = X_val.drop(columns=["price", "row_id", "is_train"])

In [320]:
# quick check
# Look at the Missing values
missing_values = X_train.isnull().sum()

print("Missing values in train:")
print(missing_values[missing_values > 0]) 
print("\n")
missing_values = X_val.isnull().sum()

print("Missing values in validation:")
print(missing_values[missing_values > 0]) 

Missing values in train:
Series([], dtype: int64)


Missing values in validation:
Series([], dtype: int64)


In [321]:
train_info = X_train.copy()
train_info["price"] = y_train
train_info["is_train"] = True
train_info = train_info.dropna()
train_info["row_id"] = np.nan
X_test["is_train"]=False
X_test["price"]=np.nan
X_test["row_id"]=X_test.index
X_test.info()
train_info.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32567 entries, 0 to 32566
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   carID           32567 non-null  int64  
 1   Brand           31918 non-null  object 
 2   model           31917 non-null  object 
 3   year            31914 non-null  float64
 4   transmission    31944 non-null  object 
 5   mileage         31878 non-null  float64
 6   fuelType        31911 non-null  object 
 7   tax             29259 non-null  float64
 8   mpg             29279 non-null  float64
 9   engineSize      31939 non-null  float64
 10  paintQuality%   31942 non-null  float64
 11  previousOwners  31970 non-null  float64
 12  hasDamage       31970 non-null  float64
 13  is_train        32567 non-null  bool   
 14  price           0 non-null      float64
 15  row_id          32567 non-null  int64  
dtypes: bool(1), float64(9), int64(2), object(4)
memory usage: 3.8+ MB
<class 'pa

In [322]:
%%time
# compute transmission
compute_transmission = pd.concat([train_info.copy(), X_test.loc[X_test["transmission"].isna()]])
transmission_values = impute_transmission(compute_transmission).loc[compute_transmission["is_train"]== False]
for idx, row in X_test[X_test['transmission'].isna()].iterrows():
    row_id = row['row_id']
    new_value = transmission_values.loc[transmission_values['row_id']==row_id, 'transmission'].iloc[0]
    X_test.at[idx, 'transmission'] = new_value

CPU times: total: 5.34 s
Wall time: 5.45 s


In [323]:
%%time
# compute fuelType
compute_fuelType = pd.concat([train_info.copy(), X_test.loc[X_test["fuelType"].isna()]])
fuelType_values = impute_fuelType(compute_fuelType).loc[compute_fuelType["is_train"]== False]
for idx, row in X_test[X_test['fuelType'].isna()].iterrows():
    row_id = row['row_id']
    new_value = fuelType_values.loc[fuelType_values['row_id']==row_id, 'fuelType'].iloc[0]
    X_test.at[idx, 'fuelType'] = new_value


CPU times: total: 6.33 s
Wall time: 6.36 s


In [324]:
%%time
# compute engineSize
compute_engineSize = pd.concat([train_info.copy(), X_test.loc[X_test["engineSize"].isna()]])
engineSize_values = impute_engineSize(compute_engineSize).loc[compute_engineSize["is_train"]== False]
for idx, row in X_test[X_test['engineSize'].isna()].iterrows():
    row_id = row['row_id']
    new_value = engineSize_values.loc[engineSize_values['row_id']==row_id, 'engineSize'].iloc[0]
    X_test.at[idx, 'engineSize'] = new_value

CPU times: total: 641 ms
Wall time: 674 ms


In [325]:
# fill has damage
X_test["hasDamage"] = X_test["hasDamage"].fillna(1)

In [326]:
%%time
# compute mpg
compute_mpg = pd.concat([train_info.copy(), X_test.loc[X_test["mpg"].isna()]])
mpg_values = impute_mpg(compute_mpg).loc[compute_mpg["is_train"]== False]
for idx, row in X_test[X_test['mpg'].isna()].iterrows():
    row_id = row['row_id']
    new_value = mpg_values.loc[mpg_values['row_id']==row_id, 'mpg'].iloc[0]
    X_test.at[idx, 'mpg'] = new_value

CPU times: total: 3.41 s
Wall time: 3.44 s


In [327]:
%%time
# compute tax
compute_tax = pd.concat([train_info.copy(), X_test.loc[X_test["tax"].isna()]])
tax_values = impute_tax(compute_tax).loc[compute_tax["is_train"]== False]
for idx, row in X_test[X_test['tax'].isna()].iterrows():
    row_id = row['row_id']
    new_value = tax_values.loc[tax_values['row_id']==row_id, 'tax'].iloc[0]
    X_test.at[idx, 'tax'] = new_value

CPU times: total: 4.62 s
Wall time: 4.67 s


In [328]:
%%time
# compute year
compute_year = pd.concat([train_info.copy(), X_test.loc[X_test["year"].isna()]])
year_values = impute_year(compute_year).loc[compute_year["is_train"]== False]
for idx, row in X_test[X_test['year'].isna()].iterrows():
    row_id = row['row_id']
    new_value = year_values.loc[year_values['row_id']==row_id, 'year'].iloc[0]
    X_test.at[idx, 'year'] = new_value

CPU times: total: 391 ms
Wall time: 404 ms


In [329]:
%%time
# compute mileage
compute_mileage = pd.concat([train_info.copy(), X_test.loc[X_test["mileage"].isna()]])
mileage_values = impute_mileage(compute_mileage).loc[compute_mileage["is_train"]== False]
for idx, row in X_test[X_test['mileage'].isna()].iterrows():
    row_id = row['row_id']
    new_value = mileage_values.loc[mileage_values['row_id']==row_id, 'mileage'].iloc[0]
    X_test.at[idx, 'mileage'] = new_value

CPU times: total: 344 ms
Wall time: 347 ms


In [330]:
%%time
# compute paintQuality%
compute_paintQuality = pd.concat([train_info.copy(), X_test.loc[X_test["paintQuality%"].isna()]])
paintQuality_values = impute_paintQuality(compute_paintQuality).loc[compute_paintQuality["is_train"]== False]
for idx, row in X_test[X_test['paintQuality%'].isna()].iterrows():
    row_id = row['row_id']
    new_value = paintQuality_values.loc[paintQuality_values['row_id']==row_id, 'paintQuality%'].iloc[0]
    X_test.at[idx, 'paintQuality%'] = new_value

CPU times: total: 375 ms
Wall time: 352 ms


In [331]:
%%time
# compute previousOwners
compute_previousOwners = pd.concat([train_info.copy(), X_test.loc[X_test["previousOwners"].isna()]])
previousOwners_values = impute_previousOwners(compute_previousOwners).loc[compute_previousOwners["is_train"]== False]
for idx, row in X_test[X_test['previousOwners'].isna()].iterrows():
    row_id = row['row_id']
    new_value = previousOwners_values.loc[previousOwners_values['row_id']==row_id, 'previousOwners'].iloc[0]
    X_test.at[idx, 'previousOwners'] = new_value

CPU times: total: 297 ms
Wall time: 298 ms


In [332]:
%%time
# compute Brand
compute_brand = pd.concat([train_info.copy(), X_test.loc[X_test["Brand"].isna()]])
brand_values = impute_brand(compute_brand).loc[compute_brand["is_train"]== False]
for idx, row in X_test[X_test['Brand'].isna()].iterrows():
    row_id = row['row_id']
    new_value = brand_values.loc[brand_values['row_id']==row_id, 'Brand'].iloc[0]
    X_test.at[idx, 'Brand'] = new_value

CPU times: total: 6.61 s
Wall time: 6.64 s


In [333]:
%%time
# compute model
compute_model = pd.concat([train_info.copy(), X_test.loc[X_test["model"].isna()]])
model_values = impute_model(compute_model).loc[compute_model["is_train"]== False]
for idx, row in X_test[X_test['model'].isna()].iterrows():
    row_id = row['row_id']
    new_value = model_values.loc[model_values['row_id']==row_id, 'model'].iloc[0]
    X_test.at[idx, 'model'] = new_value

CPU times: total: 26.5 s
Wall time: 26.7 s


In [334]:
X_test = X_test.drop(columns=["price", "row_id", "is_train"])

In [335]:
# quick check
# Look at the Missing values
missing_values = X_test.isnull().sum()

print("Missing values in test:")
print(missing_values[missing_values > 0])

Missing values in test:
Series([], dtype: int64)


# Encoding categorical variables

- We have to find out which encoding method is the best for our categorical features

In [362]:
trans_medians = X_train.groupby('transmission')['mpg'].median()
X_train['mpg_median_trans'] = X_train['transmission'].map(trans_medians)
X_val['mpg_median_trans'] = X_val['transmission'].map(trans_medians)
X_test['mpg_median_trans'] = X_test['transmission'].map(trans_medians)

In [363]:
low_card_cat = [c for c in categorical_cols if X_train[c].nunique() <= 15]
high_card_cat = [c for c in categorical_cols if c not in low_card_cat]

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
ohe.fit(X_train[low_card_cat])

def apply_ohe(df):
    ohe_arr = ohe.transform(df[low_card_cat])
    ohe_cols = ohe.get_feature_names_out(low_card_cat)
    ohe_df = pd.DataFrame(ohe_arr, columns=ohe_cols, index=df.index)
    df_rest = df.drop(columns=low_card_cat)
    return pd.concat([df_rest, ohe_df], axis=1)

x_train_enc = apply_ohe(X_train)
x_val_enc   = apply_ohe(X_val)
x_test_enc  = apply_ohe(X_test)


# print the columns after encoding
print("Columns after encoding:")
print(x_test_enc.columns.tolist())


# Print the number of columns after encoding for test val and train
print("Number of columns after encoding:")
print("x_train_enc:", x_train_enc.shape[1])
print("x_val_enc:", x_val_enc.shape[1])
print("x_test_enc:", x_test_enc.shape[1])

Columns after encoding:
['carID', 'model', 'year', 'mileage', 'tax', 'mpg', 'engineSize', 'paintQuality%', 'previousOwners', 'hasDamage', 'brand_encoded', 'model_delta_encoded', 'transmission_encoded', 'fuelType_encoded', 'mpg_median_trans', 'Brand_Audi', 'Brand_BMW', 'Brand_Ford', 'Brand_Hyundai', 'Brand_Mercedes-Benz', 'Brand_Opel', 'Brand_Skoda', 'Brand_Toyota', 'Brand_Volkswagen', 'Brand_Škoda', 'transmission_Automatic', 'transmission_Manual', 'transmission_Other', 'transmission_Semi-Auto', 'transmission_Unknown', 'fuelType_Diesel', 'fuelType_Electric', 'fuelType_Hybrid', 'fuelType_Other', 'fuelType_Petrol']
Number of columns after encoding:
x_train_enc: 34
x_val_enc: 34
x_test_enc: 35


# Feature Engineering

In [364]:
x_train_enc['mpg_diff_transmission'] = x_train_enc['mpg'] - x_train_enc['mpg_median_trans']

x_train_enc = x_train_enc.drop(columns=["mpg_median_trans"])

# age affects cars much more than the year: car_age in years
x_train_enc['car_age'] = 2020 - x_train_enc['year']
# it is important how efficent the motor is, 
x_train_enc['efficiency_ratio'] = x_train_enc['mpg'] / x_train_enc['engineSize']

x_train_enc['mileage_per_year'] = x_train_enc['mileage'] / (x_train_enc['car_age'] + 1)

#brand_popularity = x_train_enc['Brand'].value_counts(normalize=True)
#x_train_enc['brand_popularity'] = x_train_enc['Brand'].map(brand_popularity)

x_train_enc['owners_flag'] = (x_train_enc['previousOwners'] > 2).astype(int)

x_train_enc['previousOwners_sq'] = x_train_enc['previousOwners'] ** 2

x_train_enc['engine_tax_ratio'] = x_train_enc['engineSize'] / (x_train_enc['tax'] + 1)

In [365]:
x_train_enc.describe()

,year,mileage,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage,brand_encoded,model_delta_encoded,transmission_encoded,fuelType_encoded,Brand_Audi,Brand_BMW,Brand_Ford,Brand_Hyundai,Brand_Mercedes-Benz,Brand_Opel,Brand_Skoda,Brand_Toyota,Brand_Volkswagen,Brand_Škoda,transmission_Automatic,transmission_Manual,transmission_Other,transmission_Semi-Auto,transmission_Unknown,fuelType_Diesel,fuelType_Electric,fuelType_Hybrid,fuelType_Other,fuelType_Petrol,mpg_diff_transmission,car_age,efficiency_ratio,mileage_per_year,owners_flag,previousOwners_sq,engine_tax_ratio
count,56979.000000,56979.000000,56979.000000,56979.000000,56979.000000,56979.000000,56979.000000,56979.000000,56979.000000,5.697900e+04,56979.000000,56979.000000,56979.000000,56979.000000,56979.000000,56979.000000,56979.000000,56979.000000,56979.000000,56979.000000,56979.000000,56979.000000,56979.000000,56979.000000,56979.000000,56979.000000,56979.000000,56979.000000,56979.000000,56979.000000,56979.000000,56979.000000,56979.000000,56979.000000,56979.000000,56979.000000,56979.000000,56979.000000,56979.000000
mean,2017.103600,23283.774942,118.201069,54.888902,1.669537,64.656617,2.017743,0.020025,16889.092543,5.097601e-13,16889.092543,16889.092543,0.097422,0.099001,0.218203,0.044157,0.156795,0.126239,0.001018,0.061373,0.139385,0.056407,0.200565,0.561242,0.000088,0.228154,0.009951,0.415943,0.000053,0.029256,0.002176,0.552572,-0.256735,2.896400,36.775556,5287.900895,0.393303,6.099282,0.131270
std,2.176242,21547.457861,65.012960,11.372824,0.556487,20.787656,1.424089,0.140087,5276.663955,5.387250e+03,5416.080934,2109.680263,0.296534,0.298666,0.413030,0.205445,0.363610,0.332122,0.031889,0.240016,0.346351,0.230708,0.400427,0.496240,0.009367,0.419646,0.099258,0.492888,0.007256,0.168526,0.046600,0.497233,11.126439,2.176242,14.381436,4119.459567,0.488487,6.198124,0.396010
min,1996.000000,1.000000,0.000000,9.429179,1.000000,1.638913,0.000000,0.000000,10344.168219,-2.175586e+04,12204.484693,14994.771891,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-48.270821,-4.000000,2.338443,-42466.000000,0.000000,0.000000,0.003040
25%,2016.000000,7453.000000,30.000000,47.100000,1.200000,47.000000,1.000000,0.000000,12592.985764,-2.426275e+03,12204.484693,14994.771891,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-7.900000,1.000000,27.150000,2805.500000,0.000000,1.000000,0.009589
50%,2017.000000,17382.000000,145.000000,55.400000,1.600000,65.000000,2.000000,0.000000,14308.581518,-4.177218e+02,12204.484693,14994.771891,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,3.000000,33.642857,4828.571429,0.000000,4.000000,0.013245
75%,2019.000000,32357.000000,145.000000,62.800000,2.000000,82.000000,3.000000,0.000000,23008.602954,7.849704e+02,21476.692422,19233.131139,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,8.000000,4.000000,47.555556,7038.142857,1.000000,9.000000,0.032258
max,2024.000000,323000.000000,580.000000,94.100000,6.200000,125.594308,6.000000,1.000000,24500.856167,8.231802e+04,24377.936846,19381.517097,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,41.800000,24.000000,74.300000,95204.181674,1.000000,36.000000,3.822758


In [ ]:
x_val_enc['mpg_diff_transmission'] = x_val_enc['mpg'] - x_val_enc['mpg_median_trans']

x_val_enc = x_val_enc.drop(columns=["mpg_median_trans"])

# age affects cars much more than the year: car_age in years
x_val_enc['car_age'] = 2020 - x_val_enc['year']
# it is important how efficent the motor is, 
x_val_enc['efficiency_ratio'] = x_val_enc['mpg'] / x_val_enc['engineSize']

x_val_enc['mileage_per_year'] = x_val_enc['mileage'] / (x_val_enc['car_age'] + 1)

#brand_popularity = x_val_enc['Brand'].value_counts(normalize=True)
#x_val_enc['brand_popularity'] = x_val_enc['Brand'].map(brand_popularity)

x_val_enc['owners_flag'] = (x_val_enc['previousOwners'] > 2).astype(int)

x_val_enc['previousOwners_sq'] = x_val_enc['previousOwners'] ** 2

x_val_enc['engine_tax_ratio'] = x_val_enc['engineSize'] / (x_val_enc['tax'] + 1)

In [ ]:
x_test_enc['mpg_diff_transmission'] = x_test_enc['mpg'] - x_test_enc['mpg_median_trans']

x_test_enc = x_test_enc.drop(columns=["mpg_median_trans"])

# age affects cars much more than the year: car_age in years
x_test_enc['car_age'] = 2020 - x_test_enc['year']
# it is important how efficent the motor is, 
x_test_enc['efficiency_ratio'] = x_test_enc['mpg'] / x_test_enc['engineSize']

x_test_enc['mileage_per_year'] = x_test_enc['mileage'] / (x_test_enc['car_age'] + 1)

#brand_popularity = x_test_enc['Brand'].value_counts(normalize=True)
#x_test_enc['brand_popularity'] = x_test_enc['Brand'].map(brand_popularity)

x_test_enc['owners_flag'] = (x_test_enc['previousOwners'] > 2).astype(int)

x_test_enc['previousOwners_sq'] = x_test_enc['previousOwners'] ** 2

x_test_enc['engine_tax_ratio'] = x_test_enc['engineSize'] / (x_test_enc['tax'] + 1)

# Scaling

After encoding all our values are numerical, but we need to scale the features for better model performance.
The Values for mileage can be very high compared to other features, so scaling is important.



In [352]:
scaler = RobustScaler()

# numeric cols after FE (on train)
num_after_enc = x_train_enc.select_dtypes(include=["number"]).columns

# fit on TRAIN only
scaler.fit(x_train_enc[num_after_enc])

# copy
x_train_final = x_train_enc.copy()
x_val_final   = x_val_enc.copy()
x_test_final  = x_test_enc.copy()

# scale train and val on same cols
x_train_final[num_after_enc] = scaler.transform(x_train_enc[num_after_enc])
x_val_final[num_after_enc]   = scaler.transform(x_val_enc[num_after_enc])

# for test: only the intersection of cols
test_cols = [c for c in num_after_enc if c in x_test_enc.columns]
x_test_final[test_cols] = scaler.transform(x_test_enc[test_cols])

In [353]:
# Check for NaN values in each dataset
def check_nan(df, name):
    nan_cols = df.columns[df.isna().any()].tolist()
    if nan_cols:
        print(f"{name} has NaN values in columns: {nan_cols}")
    else:
        print(f"{name} has no NaN values.")


check_nan(x_train_final, "x_train_final")
check_nan(y_train.to_frame(), "y_train")
check_nan(x_val_final, "x_val_final")
check_nan(y_val.to_frame(), "y_val")
check_nan(x_test_final, "x_test_final")

x_train_final has no NaN values.
y_train has no NaN values.
x_val_final has no NaN values.
y_val has no NaN values.
x_test_final has no NaN values.


# Output Save

In [ ]:
# Save Processed Datasets
"""
Save X_train, y_train, X_val, y_val, and X_test separately.
This structure is cleaner for later model loading and avoids re-splitting.
"""

import os

output_dir = os.path.join(data_dir, "encoded_data")
os.makedirs(output_dir, exist_ok=True)

# Save feature and target sets separately
X_train.to_csv(os.path.join(output_dir, "12_X_train.csv"), index=False)
y_train.to_csv(os.path.join(output_dir, "12_y_train.csv"), index=False)

X_val.to_csv(os.path.join(output_dir, "12_X_val.csv"), index=False)
y_val.to_csv(os.path.join(output_dir, "12_y_val.csv"), index=False)

X_test.to_csv(os.path.join(output_dir, "12_X_test.csv"), index=False)

print("Processed data saved successfully (X/y separated):")
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape}, y_val: {y_val.shape}")
print(f"X_test:  {X_test.shape}")
